# 우수모델 성능 결과

In [ ]:
# ============================================================
# 0. 라이브러리
# ============================================================

import pandas as pd
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    f1_score, recall_score, precision_score,
    roc_auc_score, average_precision_score, accuracy_score
)
from xgboost import XGBClassifier
import warnings
warnings.filterwarnings("ignore")
import os


# ============================================================
# 1. 설정값
# ============================================================

TRAIN_PATH   = r'10,11,12번\train데이터\M19_도매_소매업_train.parquet'
TEST_PATH    = r'10,11,12번\test데이터\M19_도매_소매업_test.parquet'
FEATURE_PATH = r'13번.피처셀렉션\M19_도매_소매업\lasso_features_top55--45.csv'

TARGET_COL   = "부실라벨_ICR3년"
RANDOM_STATE = 42
THRESHOLD    = 0.44
YEAR_COL     = "회계년도"
ID_COLS      = ["회사명", "사업자등록번호", "회계년도"]

FOLD_VAL_YEARS = [2016, 2017, 2018, 2019, 2020, 2021]
TRAIN_START    = 2012

SAVE_DIR = r'14번. 우수모델 데이터'

# ============================================================
# 2. 데이터 로드
# ============================================================

train_full = pd.read_parquet(TRAIN_PATH)
test       = pd.read_parquet(TEST_PATH)

y_train_full = train_full[TARGET_COL]
y_test       = test[TARGET_COL]

feat_df      = pd.read_csv(FEATURE_PATH)
col_key      = "feature" if "feature" in feat_df.columns else feat_df.columns[0]
use_features = [f for f in feat_df[col_key].tolist() if f in train_full.columns]

pos_weight = (y_train_full == 0).sum() / (y_train_full == 1).sum()

print("=" * 65)
print(f"Train shape  : {train_full.shape}")
print(f"Test  shape  : {test.shape}")
print(f"피처 수       : {len(use_features)}개")
print(f"pos_weight   : {pos_weight:.4f}")
print(f"threshold    : {THRESHOLD}")
print("=" * 65)


# ============================================================
# 3. 모델 정의
# ============================================================

def make_model():
    return XGBClassifier(
        n_estimators=300, learning_rate=0.05, max_depth=4,
        subsample=0.8, colsample_bytree=0.8,
        eval_metric="aucpr",
        random_state=RANDOM_STATE, verbosity=0,
        scale_pos_weight=pos_weight
    )


# ============================================================
# 4. 평가 헬퍼
# ============================================================

def calc_metrics(y_true, y_prob, threshold=THRESHOLD):
    y_pred = (y_prob >= threshold).astype(int)
    return {
        "F1"        : f1_score(y_true, y_pred, zero_division=0),
        "Recall"    : recall_score(y_true, y_pred, zero_division=0),
        "Precision" : precision_score(y_true, y_pred, zero_division=0),
        "ROC_AUC"   : roc_auc_score(y_true, y_prob),
        "PR_AUC"    : average_precision_score(y_true, y_prob),
        "Accuracy"  : accuracy_score(y_true, y_pred),
    }


# ============================================================
# 5. Expanding Window CV
# ============================================================

print("\nExpanding Window CV")
print("=" * 65)

cv_rows      = []
val_prob_all = []   # 전체 val 확률 수집 (분포용)

for val_year in FOLD_VAL_YEARS:
    train_idx = train_full.index[
        (train_full[YEAR_COL] >= TRAIN_START) & (train_full[YEAR_COL] < val_year)
    ]
    val_idx = train_full.index[train_full[YEAR_COL] == val_year]

    if len(train_idx) == 0 or len(val_idx) == 0:
        print(f"  [경고] val_year={val_year} 데이터 없음 → skip")
        continue

    X_fold_train = train_full.loc[train_idx, use_features]
    y_fold_train = train_full.loc[train_idx, TARGET_COL]
    X_fold_val   = train_full.loc[val_idx,   use_features]
    y_fold_val   = train_full.loc[val_idx,   TARGET_COL]

    imputer      = SimpleImputer(strategy="median")
    X_fold_train = pd.DataFrame(imputer.fit_transform(X_fold_train), columns=use_features)
    X_fold_val   = pd.DataFrame(imputer.transform(X_fold_val),       columns=use_features)

    model = make_model()
    model.fit(X_fold_train, y_fold_train)
    y_prob_val = model.predict_proba(X_fold_val)[:, 1]

    m = calc_metrics(y_fold_val, y_prob_val)
    print(f"  Val {val_year} | F1={m['F1']:.4f} Recall={m['Recall']:.4f} "
          f"Precision={m['Precision']:.4f} ROC_AUC={m['ROC_AUC']:.4f} PR_AUC={m['PR_AUC']:.4f}")

    cv_rows.append({"Val_Year": val_year, **m})
    val_prob_all.append(pd.DataFrame({
        "Val_Year"   : val_year,
        "y_true"     : y_fold_val.values,
        "y_prob"     : y_prob_val,
        "y_pred"     : (y_prob_val >= THRESHOLD).astype(int),
    }))

cv_df = pd.DataFrame(cv_rows).round(4)
cv_mean = cv_df[["F1","Recall","Precision","ROC_AUC","PR_AUC","Accuracy"]].mean()

print(f"\n  {'─'*55}")
print(f"  CV 평균 | F1={cv_mean['F1']:.4f} Recall={cv_mean['Recall']:.4f} "
      f"Precision={cv_mean['Precision']:.4f} ROC_AUC={cv_mean['ROC_AUC']:.4f} "
      f"PR_AUC={cv_mean['PR_AUC']:.4f}")


# ============================================================
# 6. Test 평가 (전체 train 재학습)
# ============================================================

print(f"\n{'='*65}")
print("Test 평가")
print("=" * 65)

imputer_final = SimpleImputer(strategy="median")
X_train_all   = pd.DataFrame(
    imputer_final.fit_transform(train_full[use_features]), columns=use_features
)
X_test_imp    = pd.DataFrame(
    imputer_final.transform(test[use_features]), columns=use_features
)

final_model = make_model()
final_model.fit(X_train_all, y_train_full)
y_prob_test = final_model.predict_proba(X_test_imp)[:, 1]
y_pred_test = (y_prob_test >= THRESHOLD).astype(int)

test_metrics = calc_metrics(y_test, y_prob_test)
print(f"  Test    | F1={test_metrics['F1']:.4f} Recall={test_metrics['Recall']:.4f} "
      f"Precision={test_metrics['Precision']:.4f} ROC_AUC={test_metrics['ROC_AUC']:.4f} "
      f"PR_AUC={test_metrics['PR_AUC']:.4f}")

print(f"\n  [Gap = Test - CV평균]")
for col in ["F1","Recall","Precision","ROC_AUC","PR_AUC","Accuracy"]:
    gap = test_metrics[col] - cv_mean[col]
    print(f"    {col:<12} : {gap:+.4f}")


# ============================================================
# 7. 확률 분포 시각화 저장
# ============================================================

import matplotlib.gridspec as gridspec
from matplotlib.patches import Patch
from matplotlib.lines import Line2D

# ── 폰트/스타일 설정 ─────────────────────────────────────
plt.rcParams.update({
    "font.family"       : "DejaVu Sans",
    "axes.spines.top"   : False,
    "axes.spines.right" : False,
    "axes.grid"         : True,
    "grid.color"        : "#E5E5E5",
    "grid.linewidth"    : 0.7,
    "axes.facecolor"    : "#FAFAFA",
    "figure.facecolor"  : "white",
})

COLOR_NEG   = "#2F6EBA"   # Normal(0) — 파랑
COLOR_POS   = "#D94F3D"   # Distress(1) — 빨강
COLOR_THR   = "#F5A623"   # threshold — 주황
ALPHA_HIST  = 0.72
BINS        = 45

val_prob_df = pd.concat(val_prob_all, ignore_index=True)

# ── 레이아웃: 2열 메인 + 하단 메트릭 테이블 ──────────────
fig = plt.figure(figsize=(16, 10))
gs  = gridspec.GridSpec(
    2, 2,
    height_ratios=[3.2, 1],
    hspace=0.42, wspace=0.32,
    left=0.07, right=0.97, top=0.91, bottom=0.05
)

ax_cv   = fig.add_subplot(gs[0, 0])
ax_test = fig.add_subplot(gs[0, 1])
ax_tbl  = fig.add_subplot(gs[1, :])
ax_tbl.axis("off")

# ── 공통 플롯 함수 ────────────────────────────────────────
def plot_dist(ax, y_true, y_prob, title, n_total):
    arr0 = y_prob[np.array(y_true) == 0]
    arr1 = y_prob[np.array(y_true) == 1]

    counts0, edges0 = np.histogram(arr0, bins=BINS, range=(0, 1))
    counts1, edges1 = np.histogram(arr1, bins=BINS, range=(0, 1))

    ax.bar(edges0[:-1], counts0, width=np.diff(edges0),
           align="edge", color=COLOR_NEG, alpha=ALPHA_HIST, label="Normal (0)", zorder=3)
    ax.bar(edges1[:-1], counts1, width=np.diff(edges1),
           align="edge", color=COLOR_POS, alpha=ALPHA_HIST, label="Distress (1)", zorder=3)

    # threshold 수직선
    ax.axvline(THRESHOLD, color=COLOR_THR, linestyle="--",
               linewidth=1.8, zorder=5, label=f"Threshold = {THRESHOLD}")

    # 음영 — threshold 우측 위험 영역
    ax.axvspan(THRESHOLD, 1.0, alpha=0.06, color=COLOR_POS, zorder=2)

    # 통계 annotation
    n0, n1 = len(arr0), len(arr1)
    ir = n1 / n0 if n0 > 0 else float("nan")
    above_thr = (y_prob >= THRESHOLD).sum()
    stats_txt = (
        f"N={n_total:,}  |  Normal={n0:,}  Distress={n1:,}\n"
        f"Imbalance ratio = {ir:.3f}  |  Predicted Positive = {above_thr:,}"
    )
    ax.text(0.98, 0.97, stats_txt,
            transform=ax.transAxes, fontsize=8.2,
            va="top", ha="right",
            bbox=dict(boxstyle="round,pad=0.4", fc="white", ec="#CCCCCC", alpha=0.85))

    ax.set_title(title, fontsize=12, fontweight="bold", pad=10)
    ax.set_xlabel("Predicted Probability", fontsize=9.5)
    ax.set_ylabel("Count", fontsize=9.5)
    ax.set_xlim(0, 1)
    ax.tick_params(labelsize=8.5)

    legend_elems = [
        Patch(facecolor=COLOR_NEG, alpha=ALPHA_HIST, label="Normal (0)"),
        Patch(facecolor=COLOR_POS, alpha=ALPHA_HIST, label="Distress (1)"),
        Line2D([0], [0], color=COLOR_THR, linestyle="--", linewidth=1.8,
               label=f"Threshold = {THRESHOLD}"),
    ]
    ax.legend(handles=legend_elems, fontsize=8.5, framealpha=0.9,
              loc="upper left", edgecolor="#CCCCCC")

plot_dist(ax_cv,
          val_prob_df["y_true"].values,
          val_prob_df["y_prob"].values,
          "Expanding Window CV — Predicted Probability Distribution",
          n_total=len(val_prob_df))

plot_dist(ax_test,
          y_test.values,
          y_prob_test,
          "Hold-out Test — Predicted Probability Distribution",
          n_total=len(y_test))

# ── 하단 메트릭 테이블 ────────────────────────────────────
metrics_order = ["F1", "Recall", "Precision", "ROC_AUC", "PR_AUC", "Accuracy"]
col_labels    = ["Split"] + metrics_order

cv_row   = ["CV Mean"] + [f"{float(cv_mean[m]):.4f}"   for m in metrics_order]
test_row = ["Test"]    + [f"{test_metrics[m]:.4f}"     for m in metrics_order]
gap_row  = ["Gap (Test − CV)"] + [
    f"{test_metrics[m] - float(cv_mean[m]):+.4f}" for m in metrics_order
]

table_data = [cv_row, test_row, gap_row]

tbl = ax_tbl.table(
    cellText=table_data,
    colLabels=col_labels,
    cellLoc="center",
    loc="center",
)
tbl.auto_set_font_size(False)
tbl.set_fontsize(9)
tbl.scale(1, 1.7)

# 헤더 스타일
for j in range(len(col_labels)):
    tbl[(0, j)].set_facecolor("#2F4F7F")
    tbl[(0, j)].set_text_props(color="white", fontweight="bold")

# 행 배경
row_colors = ["#EEF3FA", "#FAFAFA", "#FFF4EE"]
for i, rc in enumerate(row_colors, start=1):
    for j in range(len(col_labels)):
        tbl[(i, j)].set_facecolor(rc)

# Gap 행 — 음수(빨강) / 양수(초록) 색상
for j, m in enumerate(metrics_order, start=1):
    gap_val = test_metrics[m] - float(cv_mean[m])
    color   = "#C0392B" if gap_val < -0.02 else ("#27AE60" if gap_val > 0.02 else "#555555")
    tbl[(3, j)].set_text_props(color=color, fontweight="bold")

ax_tbl.set_title("Performance Summary", fontsize=10,
                 fontweight="bold", pad=6, loc="left")

# ── 전체 제목 ─────────────────────────────────────────────
fig.suptitle(
    "XGBoost  │  Class Weight (scale_pos_weight)  │  M19 도매·소매업",
    fontsize=13.5, fontweight="bold", y=0.975
)

plt.savefig(
    os.path.join(SAVE_DIR, "XGB_CW_prob_distribution.png"),
    dpi=180, bbox_inches="tight"
)
plt.close()
print(f"\n  확률 분포 이미지 저장 완료")



# ============================================================
# 8. Test 전체 행 예측 결과 저장
# ============================================================

# ID 컬럼 붙이기
test_id_cols = [c for c in ID_COLS if c in test.columns]
test_pred_df = test[test_id_cols].copy().reset_index(drop=True)
test_pred_df["y_true"]       = y_test.values
test_pred_df["y_prob"]       = y_prob_test.round(4)
test_pred_df["y_pred"]       = y_pred_test
test_pred_df["correct"]      = (test_pred_df["y_true"] == test_pred_df["y_pred"]).astype(int)
test_pred_df["error_type"]   = "TN"
test_pred_df.loc[(test_pred_df["y_true"]==1) & (test_pred_df["y_pred"]==1), "error_type"] = "TP"
test_pred_df.loc[(test_pred_df["y_true"]==1) & (test_pred_df["y_pred"]==0), "error_type"] = "FN"
test_pred_df.loc[(test_pred_df["y_true"]==0) & (test_pred_df["y_pred"]==1), "error_type"] = "FP"

print(f"\n  [Test 예측 분포]")
print(test_pred_df["error_type"].value_counts().to_string())


# ============================================================
# 9. 저장
# ============================================================

# CV fold별 성능
import os
SAVE_DIR = r'15번. 우수모델 데이터'
os.makedirs(SAVE_DIR, exist_ok=True)

cv_df.to_csv(os.path.join(SAVE_DIR, "XGB_CW_CV_results.csv"), index=False, encoding="utf-8-sig")

# CV평균 + Test + Gap 요약
summary = {"Method": "ClassWeight", "FeatureFile": FEATURE_PATH.split("\\")[-1],
           "N_Features": len(use_features), "threshold": THRESHOLD}
for col in ["F1","Recall","Precision","ROC_AUC","PR_AUC","Accuracy"]:
    summary[f"CV_Val_{col}"] = round(float(cv_mean[col]), 4)
    summary[f"Test_{col}"]   = round(test_metrics[col], 4)
    summary[f"Gap_{col}"]    = round(test_metrics[col] - float(cv_mean[col]), 4)
pd.DataFrame([summary]).to_csv(os.path.join(SAVE_DIR, "XGB_CW_summary.csv"), index=False, encoding="utf-8-sig")

# Test 전체 행 예측 결과
test_pred_df.to_csv(os.path.join(SAVE_DIR, "XGB_CW_test_predictions.csv"), index=False, encoding="utf-8-sig")

# CV val 전체 확률 분포 raw
val_prob_df.to_csv(os.path.join(SAVE_DIR, "XGB_CW_val_prob_distribution.csv"), index=False, encoding="utf-8-sig")



Train shape  : (28111, 256)
Test  shape  : (11797, 256)
피처 수       : 45개
pos_weight   : 25.6455
threshold    : 0.44

Expanding Window CV
  Val 2016 | F1=0.4153 Recall=0.6622 Precision=0.3025 ROC_AUC=0.9536 PR_AUC=0.4580
  Val 2017 | F1=0.4353 Recall=0.5286 Precision=0.3700 ROC_AUC=0.9568 PR_AUC=0.3959
  Val 2018 | F1=0.4589 Recall=0.7444 Precision=0.3317 ROC_AUC=0.9587 PR_AUC=0.3685
  Val 2019 | F1=0.4099 Recall=0.7830 Precision=0.2776 ROC_AUC=0.9530 PR_AUC=0.4657
  Val 2020 | F1=0.3620 Recall=0.5900 Precision=0.2611 ROC_AUC=0.9479 PR_AUC=0.3244
  Val 2021 | F1=0.4791 Recall=0.8651 Precision=0.3313 ROC_AUC=0.9717 PR_AUC=0.4840

  ───────────────────────────────────────────────────────
  CV 평균 | F1=0.4267 Recall=0.6955 Precision=0.3124 ROC_AUC=0.9569 PR_AUC=0.4161

Test 평가
  Test    | F1=0.4203 Recall=0.9119 Precision=0.2731 ROC_AUC=0.9597 PR_AUC=0.4525

  [Gap = Test - CV평균]
    F1           : -0.0064
    Recall       : +0.2163
    Precision    : -0.0393
    ROC_AUC      : +0.0027
    

# SHAP

## 전역

In [4]:
# ============================================================
# [SHAP 단독 실행 시 필요한 변수 재선언]
# ============================================================

import pandas as pd
import numpy as np
import os
import shap
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
from sklearn.impute import SimpleImputer
from sklearn.inspection import permutation_importance
from xgboost import XGBClassifier

import warnings
warnings.filterwarnings("ignore")

# ── ★ 한글 폰트 설정 ──────────────────────────────────────
import platform

if platform.system() == "Windows":
    plt.rcParams["font.family"] = "Malgun Gothic"       # 윈도우
elif platform.system() == "Darwin":
    plt.rcParams["font.family"] = "AppleGothic"         # 맥
else:
    # Linux: 나눔고딕 설치 필요 (pip install koreanize-matplotlib)
    try:
        import koreanize_matplotlib
    except ImportError:
        pass

plt.rcParams["axes.unicode_minus"] = False              # 마이너스 기호 깨짐 방지

# ── 경로 설정 ─────────────────────────────────────────────
TRAIN_PATH = r'10,11,12번\train데이터\M19_도매_소매업_train.parquet'
TEST_PATH  = r'10,11,12번\test데이터\M19_도매_소매업_test.parquet'

TARGET_COL    = "부실라벨_ICR3년"
RANDOM_STATE  = 42
THRESHOLD     = 0.44
SHAP_SAVE_DIR = r"16번. SHAP\전역"      # ★ 경로 변경
os.makedirs(SHAP_SAVE_DIR, exist_ok=True)

# ── 데이터 로드 ───────────────────────────────────────────
train_full = pd.read_parquet(TRAIN_PATH)
test       = pd.read_parquet(TEST_PATH)

y_train_full = train_full[TARGET_COL]
y_test       = test[TARGET_COL]

pos_weight = (y_train_full == 0).sum() / (y_train_full == 1).sum()

# ── 피처 파일 ─────────────────────────────────────────────
feature_files = {
    "top55_dedup45": r"13번.피처셀렉션\M19_도매_소매업\lasso_features_top55--45.csv",
}

# ── feature_map 재구성 ────────────────────────────────────
feature_map = {}
for feature_name, feature_path in feature_files.items():
    df_feat        = pd.read_csv(feature_path)
    col_key        = "feature" if "feature" in df_feat.columns else df_feat.columns[0]
    raw_features   = df_feat[col_key].tolist()
    valid_features = [f for f in raw_features if f in train_full.columns]
    feature_map[feature_name] = valid_features
    print(f"[{feature_name}]  피처: {len(valid_features)}개")

print(f"pos_weight: {pos_weight:.4f}")
print("=" * 70)


# ============================================================
# 모델 정의
# ============================================================

def make_final_model():
    return XGBClassifier(
        n_estimators=300, learning_rate=0.05, max_depth=4,
        subsample=0.8, colsample_bytree=0.8,
        eval_metric="aucpr",
        random_state=RANDOM_STATE, verbosity=0,
        scale_pos_weight=pos_weight
    )


# ============================================================
# 피처셋별 SHAP + Permutation 루프
# ============================================================

for feature_name, use_features in feature_map.items():

    feature_file_name = feature_files[feature_name].split("\\")[-1]
    print(f"\n{'='*70}")
    print(f"[SHAP/Permutation] Feature Set: {feature_name}  |  피처 수: {len(use_features)}개")
    print(f"{'='*70}")

    # ── 전체 train 재학습 ─────────────────────────────────────
    imputer_shap = SimpleImputer(strategy="median")
    X_train_shap = pd.DataFrame(
        imputer_shap.fit_transform(train_full[use_features]),
        columns=use_features
    )
    X_test_shap  = pd.DataFrame(
        imputer_shap.transform(test[use_features]),
        columns=use_features
    )

    model_shap = make_final_model()
    model_shap.fit(X_train_shap, y_train_full)

    # ── ★ 서브폴더: 16번. SHAP/전역/피처셋명 ─────────────────
    save_sub = os.path.join(SHAP_SAVE_DIR, feature_name)
    os.makedirs(save_sub, exist_ok=True)


    # ── [A-1] SHAP 계산 ───────────────────────────────────────
    print(f"  SHAP 계산 중...")
    explainer   = shap.TreeExplainer(model_shap)
    shap_values = explainer.shap_values(X_test_shap)

    shap_df = pd.DataFrame(shap_values, columns=use_features)
    shap_df.to_csv(
        os.path.join(save_sub, f"shap_values_{feature_name}.csv"),
        index=False, encoding="utf-8-sig"
    )

    # ── [A-2] Bar Plot ────────────────────────────────────────
    plt.figure(figsize=(10, max(6, len(use_features) * 0.3)))
    shap.summary_plot(shap_values, X_test_shap, plot_type="bar",
                      show=False, max_display=20)
    plt.title(f"SHAP Global Feature Importance (Bar)\n{feature_name}", fontsize=12)
    plt.tight_layout()
    plt.savefig(
        os.path.join(save_sub, f"shap_bar_{feature_name}.png"),
        dpi=150, bbox_inches="tight"
    )
    plt.close()

    # ── [A-3] Beeswarm Plot ───────────────────────────────────
    plt.figure(figsize=(10, max(6, len(use_features) * 0.3)))
    shap.summary_plot(shap_values, X_test_shap, plot_type="dot",
                      show=False, max_display=20)
    plt.title(f"SHAP Beeswarm Plot\n{feature_name}", fontsize=12)
    plt.tight_layout()
    plt.savefig(
        os.path.join(save_sub, f"shap_beeswarm_{feature_name}.png"),
        dpi=150, bbox_inches="tight"
    )
    plt.close()

    # ── [A-4] SHAP 중요도 CSV ─────────────────────────────────
    shap_importance = pd.DataFrame({
        "Feature"       : use_features,
        "mean_abs_SHAP" : np.abs(shap_values).mean(axis=0)
    }).sort_values("mean_abs_SHAP", ascending=False).reset_index(drop=True)
    shap_importance["Rank"] = shap_importance.index + 1
    shap_importance.to_csv(
        os.path.join(save_sub, f"shap_importance_{feature_name}.csv"),
        index=False, encoding="utf-8-sig"
    )
    print(f"  [SHAP Top 10]\n{shap_importance.head(10).to_string(index=False)}")


    # ── [B-1] Permutation Importance ─────────────────────────
    print(f"\n  Permutation Importance 계산 중...")
    perm_result = permutation_importance(
        model_shap, X_test_shap, y_test,
        n_repeats=30, random_state=RANDOM_STATE,
        scoring="average_precision", n_jobs=-1
    )

    perm_df = pd.DataFrame({
        "Feature"   : use_features,
        "Perm_Mean" : perm_result.importances_mean,
        "Perm_Std"  : perm_result.importances_std,
    }).sort_values("Perm_Mean", ascending=False).reset_index(drop=True)
    perm_df["Rank"] = perm_df.index + 1
    perm_df.to_csv(
        os.path.join(save_sub, f"permutation_importance_{feature_name}.csv"),
        index=False, encoding="utf-8-sig"
    )
    print(f"  [Permutation Top 10]\n{perm_df.head(10).to_string(index=False)}")

    # ── [B-2] Permutation Bar Plot ────────────────────────────
    top_n    = min(20, len(use_features))
    perm_top = perm_df.head(top_n).sort_values("Perm_Mean", ascending=True)

    fig, ax = plt.subplots(figsize=(10, max(6, top_n * 0.4)))
    ax.barh(
        perm_top["Feature"], perm_top["Perm_Mean"],
        xerr=perm_top["Perm_Std"],
        color="#2F6EBA", alpha=0.8,
        error_kw=dict(ecolor="#555555", capsize=3)
    )
    ax.axvline(0, color="red", linestyle="--", linewidth=1.0)
    ax.set_xlabel("Mean decrease in PR_AUC", fontsize=10)
    ax.set_title(f"Permutation Importance (Top {top_n})\n{feature_name}", fontsize=12)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    plt.tight_layout()
    plt.savefig(
        os.path.join(save_sub, f"permutation_bar_{feature_name}.png"),
        dpi=150, bbox_inches="tight"
    )
    plt.close()

    # ── [C] SHAP vs Permutation 순위 비교 CSV ─────────────────
    compare_df = shap_importance[["Rank", "Feature", "mean_abs_SHAP"]].rename(
        columns={"Rank": "SHAP_Rank"}
    ).merge(
        perm_df[["Feature", "Perm_Mean", "Rank"]].rename(
            columns={"Rank": "Perm_Rank"}
        ),
        on="Feature", how="inner"
    )
    compare_df["Rank_Diff"] = (
        compare_df["SHAP_Rank"] - compare_df["Perm_Rank"]
    ).abs()
    compare_df = compare_df.sort_values("SHAP_Rank").reset_index(drop=True)
    compare_df.to_csv(
        os.path.join(save_sub, f"shap_vs_permutation_{feature_name}.csv"),
        index=False, encoding="utf-8-sig"
    )
    print(f"\n  → {save_sub} 저장 완료")


print("\n" + "=" * 70)
print("SHAP / Permutation Importance 분석 전체 완료")
print(f"  저장 위치: {SHAP_SAVE_DIR}")
print("=" * 70)

[top55_dedup45]  피처: 45개
pos_weight: 25.6455

[SHAP/Permutation] Feature Set: top55_dedup45  |  피처 수: 45개
  SHAP 계산 중...
  [SHAP Top 10]
              Feature  mean_abs_SHAP  Rank
        총자본영업이익률_diff       2.394883     1
              금융비용부담률       0.984711     2
                ROA변화       0.689847     3
            ROIC_diff       0.489343     4
매출액순이익률_diff_industry       0.436582     5
                자본잠식률       0.333775     6
            ROA_ratio       0.282422     7
           순이익률_ratio       0.232012     8
       영업CF_유동부채_diff       0.187151     9
 차입금의존도_diff_industry       0.133473    10

  Permutation Importance 계산 중...
  [Permutation Top 10]
                 Feature  Perm_Mean  Perm_Std  Rank
           총자본영업이익률_diff   0.206601  0.013227     1
                   ROA변화   0.199079  0.009883     2
                 금융비용부담률   0.122121  0.011156     3
                    부채비율   0.048501  0.003257     4
                   자본잠식률   0.034000  0.004298     5
          비유동장기적합률_ra

In [7]:
# ============================================================
# SHAP vs Permutation 통합 분석 CSV 생성 (피처 범주 추가)
# ============================================================

import pandas as pd
import numpy as np
import os

SHAP_SAVE_DIR = r"16번. SHAP\전역"

# ============================================================
# ★ 피처 범주 사전 정의
# ============================================================

FEATURE_CATEGORY = {
    # 안정성 (Solvency)
    "자본잠식률"                        : "안정성 (Solvency)",
    "비유동장기적합률_ratio"             : "안정성 (Solvency)",
    "차입금의존도_diff_industry"         : "안정성 (Solvency)",
    "부채비율"                          : "안정성 (Solvency)",
    "자기자본비율_diff_industry"         : "안정성 (Solvency)",
    "부채비율변화"                       : "안정성 (Solvency)",
    "장기부채의존도"                     : "안정성 (Solvency)",
    "유동비율변화_diff"                  : "안정성 (Solvency)",
    "유동비율_ratio"                     : "안정성 (Solvency)",
    "장기부채비율"                       : "안정성 (Solvency)",
    "유보율_diff"                        : "안정성 (Solvency)",
    "순운전자본비율_ratio"               : "안정성 (Solvency)",
    "순운전자본대총자본_ratio_industry"  : "안정성 (Solvency)",

    # 수익성 (Profitability)
    "총자본영업이익률_diff"              : "수익성 (Profitability)",
    "금융비용부담률"                     : "수익성 (Profitability)",
    "ROA변화"                           : "수익성 (Profitability)",
    "매출액순이익률_diff_industry"       : "수익성 (Profitability)",
    "ROIC_diff"                         : "수익성 (Profitability)",
    "ROA_ratio"                         : "수익성 (Profitability)",
    "순이익률_ratio"                     : "수익성 (Profitability)",
    "현금ROA"                           : "수익성 (Profitability)",
    "매출총이익률_diff"                  : "수익성 (Profitability)",
    "매출원가율"                         : "수익성 (Profitability)",
    "ROE_diff"                          : "수익성 (Profitability)",
    "현금ROE_ratio"                     : "수익성 (Profitability)",

    # 성장성 (Growth)
    "매출액증가율"                       : "성장성 (Growth)",
    "순이익증가율_diff"                  : "성장성 (Growth)",
    "유형자산증가율_ratio_industry"      : "성장성 (Growth)",
    "총자산증가율_diff_industry"         : "성장성 (Growth)",
    "자기자본증가율"                     : "성장성 (Growth)",

    # 활동성 (Activity)
    "비유동자산회전율_ratio"             : "활동성 (Activity)",
    "유형자산회전율_diff"                : "활동성 (Activity)",
    "매출채권회전율_ratio_industry"      : "활동성 (Activity)",
    "순운전자본회전율_diff"              : "활동성 (Activity)",
    "유동자산회전율"                     : "활동성 (Activity)",
    "총자산회전율_ratio"                 : "활동성 (Activity)",
    "재고자산보유기간_ratio"             : "활동성 (Activity)",
    "매입채무지급기간_diff"              : "활동성 (Activity)",

    # 현금흐름 (Cash Flow)
    "영업CF_유동부채_diff"               : "현금흐름 (Cash Flow)",
    "영업CF_총부채_diff"                 : "현금흐름 (Cash Flow)",
    "FCF_총자산_ratio"                   : "현금흐름 (Cash Flow)",
    "영업현금흐름비율"                   : "현금흐름 (Cash Flow)",
    "감가상각비율"                       : "현금흐름 (Cash Flow)",

    # 기타
    "업력"                              : "기타 (Other)",
    "유형자산비율"                       : "기타 (Other)",
}

CATEGORY_ORDER = {
    "안정성 (Solvency)"        : 1,
    "수익성 (Profitability)"   : 2,
    "성장성 (Growth)"          : 3,
    "활동성 (Activity)"        : 4,
    "현금흐름 (Cash Flow)"     : 5,
    "기타 (Other)"             : 6,
    "미분류"                   : 7,
}


# ============================================================
# 피처셋별 루프
# ============================================================

for feature_name in feature_map.keys():

    save_sub = os.path.join(SHAP_SAVE_DIR, feature_name)

    # ── 파일 로드 ─────────────────────────────────────────
    shap_imp = pd.read_csv(
        os.path.join(save_sub, f"shap_importance_{feature_name}.csv")
    )
    perm_imp = pd.read_csv(
        os.path.join(save_sub, f"permutation_importance_{feature_name}.csv")
    )

    n_features = len(shap_imp)

    # ── 병합 ──────────────────────────────────────────────
    df = shap_imp[["Rank", "Feature", "mean_abs_SHAP"]].rename(
        columns={"Rank": "SHAP_Rank"}
    ).merge(
        perm_imp[["Feature", "Perm_Mean", "Perm_Std", "Rank"]].rename(
            columns={"Rank": "Perm_Rank"}
        ),
        on="Feature", how="inner"
    )

    # ── 정규화 순위 ───────────────────────────────────────
    df["SHAP_Rank_Norm"] = (df["SHAP_Rank"] - 1) / (n_features - 1)
    df["Perm_Rank_Norm"] = (df["Perm_Rank"] - 1) / (n_features - 1)

    # ── 순위 차이 ─────────────────────────────────────────
    df["Rank_Diff"] = (df["SHAP_Rank"] - df["Perm_Rank"]).abs()

    # ── 통합 중요도 점수 ──────────────────────────────────
    df["Combined_Score"] = (df["SHAP_Rank_Norm"] + df["Perm_Rank_Norm"]) / 2
    df["Combined_Rank"]  = df["Combined_Score"].rank(method="min").astype(int)

    # ── 피처 유형 분류 ────────────────────────────────────
    top_n    = max(3, int(n_features * 0.3))
    shap_top = set(df.nsmallest(top_n, "SHAP_Rank")["Feature"])
    perm_top = set(df.nsmallest(top_n, "Perm_Rank")["Feature"])

    def classify(row):
        in_shap = row["Feature"] in shap_top
        in_perm = row["Feature"] in perm_top
        if in_shap and in_perm:
            return "★ 핵심피처 (SHAP+Perm 모두 높음)"
        elif in_shap and not in_perm:
            return "△ 대체가능 (SHAP 높음, Perm 낮음)"
        elif not in_shap and in_perm:
            return "▲ 상호작용 (Perm 높음, SHAP 낮음)"
        else:
            return "- 일반피처"

    df["Feature_Type"] = df.apply(classify, axis=1)

    # ── 불일치 레벨 ───────────────────────────────────────
    def rank_diff_level(diff):
        if diff <= 3:
            return "일치"
        elif diff <= 8:
            return "소폭 불일치"
        else:
            return "대폭 불일치"

    df["Consistency"] = df["Rank_Diff"].apply(rank_diff_level)

    # ── 범주 컬럼 추가 ────────────────────────────────────
    df["Category"]       = df["Feature"].map(FEATURE_CATEGORY).fillna("미분류")
    df["Category_Order"] = df["Category"].map(CATEGORY_ORDER)

    # ── 컬럼 순서 정리 & 정렬 ─────────────────────────────
    df = df[[
        "Combined_Rank",
        "Feature",
        "Category",
        "Feature_Type",
        "SHAP_Rank", "mean_abs_SHAP",
        "Perm_Rank", "Perm_Mean", "Perm_Std",
        "Rank_Diff", "Consistency",
        "Combined_Score",
        "Category_Order",
    ]].sort_values("Combined_Rank").reset_index(drop=True)

    # ── 저장 ──────────────────────────────────────────────
    out_path = os.path.join(save_sub, f"feature_analysis_{feature_name}.csv")
    df.to_csv(out_path, index=False, encoding="utf-8-sig")

    # ── 요약 출력 ─────────────────────────────────────────
    print(f"\n{'='*70}")
    print(f"[{feature_name}]  피처 분석 완료  →  {out_path}")
    print(f"{'='*70}")

    # 피처 유형 분포
    print(f"\n  [피처 유형 분포]")
    for t, c in df["Feature_Type"].value_counts().items():
        print(f"    {t} : {c}개")

    # ── ★ 범주별 분포 (수정된 부분) ──────────────────────
    print(f"\n  [범주별 분포]")
    cat_order_df = df[["Category", "Category_Order"]].drop_duplicates()

    cat_summary = (
        df.groupby("Category")
        .agg(
            피처수             = ("Feature",      "count"),
            평균_Combined_Rank = ("Combined_Rank", "mean"),
            핵심피처수         = ("Feature_Type",
                                  lambda x: (x == "★ 핵심피처 (SHAP+Perm 모두 높음)").sum())
        )
        .reset_index()
        .merge(cat_order_df, on="Category", how="left")   # ★ merge 먼저
        .sort_values("Category_Order")                    # ★ 그 다음 정렬
        .reset_index(drop=True)
    )
    print(cat_summary[[
        "Category", "피처수", "평균_Combined_Rank", "핵심피처수"
    ]].to_string(index=False))

    # 핵심피처 목록
    print(f"\n  [★ 핵심피처 목록 (SHAP+Perm 모두 상위 {top_n}위 이내)]")
    core = df[df["Feature_Type"] == "★ 핵심피처 (SHAP+Perm 모두 높음)"]
    if len(core) > 0:
        print(core[[
            "Combined_Rank", "Feature", "Category",
            "SHAP_Rank", "Perm_Rank", "Rank_Diff"
        ]].to_string(index=False))
    else:
        print("    없음")

    # 미분류 피처 경고
    unclassified = df[df["Category"] == "미분류"]
    if len(unclassified) > 0:
        print(f"\n  [⚠️ 미분류 피처 ({len(unclassified)}개) — FEATURE_CATEGORY 사전에 추가 필요]")
        print(unclassified[["Feature", "SHAP_Rank", "Perm_Rank"]].to_string(index=False))

    # 대폭 불일치 피처
    print(f"\n  [대폭 불일치 피처 (Rank_Diff > 8)]")
    mismatch = df[df["Consistency"] == "대폭 불일치"].sort_values(
        "Rank_Diff", ascending=False
    )
    if len(mismatch) > 0:
        print(mismatch[[
            "Feature", "Category", "Feature_Type",
            "SHAP_Rank", "Perm_Rank", "Rank_Diff"
        ]].to_string(index=False))
    else:
        print("    없음")

print("\n" + "=" * 70)
print("전체 피처 분석 CSV 저장 완료")
print("=" * 70)


[top55_dedup45]  피처 분석 완료  →  16번. SHAP\전역\top55_dedup45\feature_analysis_top55_dedup45.csv

  [피처 유형 분포]
    - 일반피처 : 28개
    ★ 핵심피처 (SHAP+Perm 모두 높음) : 9개
    ▲ 상호작용 (Perm 높음, SHAP 낮음) : 4개
    △ 대체가능 (SHAP 높음, Perm 낮음) : 4개

  [범주별 분포]
           Category  피처수  평균_Combined_Rank  핵심피처수
     안정성 (Solvency)   13         23.307692      1
수익성 (Profitability)   12         19.666667      6
       성장성 (Growth)    5         23.200000      1
     활동성 (Activity)    8         25.625000      0
   현금흐름 (Cash Flow)    5         25.400000      1
         기타 (Other)    2         21.500000      0

  [★ 핵심피처 목록 (SHAP+Perm 모두 상위 13위 이내)]
 Combined_Rank               Feature            Category  SHAP_Rank  Perm_Rank  Rank_Diff
             1         총자본영업이익률_diff 수익성 (Profitability)          1          1          0
             2               금융비용부담률 수익성 (Profitability)          2          3          1
             2                 ROA변화 수익성 (Profitability)          3          2          1
          

## 지역

In [9]:
# ============================================================
# [단독 실행용 변수 재선언]
# ============================================================

import shap
import matplotlib
import matplotlib as mpl
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
import os
import platform
from sklearn.impute import SimpleImputer
from xgboost import XGBClassifier
import warnings
warnings.filterwarnings("ignore")

# ── 한글 폰트 ─────────────────────────────────────────────
if platform.system() == "Windows":
    plt.rcParams["font.family"]        = "Malgun Gothic"
    mpl.rcParams["font.family"]        = "Malgun Gothic"
elif platform.system() == "Darwin":
    plt.rcParams["font.family"]        = "AppleGothic"
    mpl.rcParams["font.family"]        = "AppleGothic"
plt.rcParams["axes.unicode_minus"] = False
mpl.rcParams["axes.unicode_minus"] = False

# ── 경로 ──────────────────────────────────────────────────
TRAIN_PATH    = r'10,11,12번\train데이터\M19_도매_소매업_train.parquet'
TEST_PATH     = r'10,11,12번\test데이터\M19_도매_소매업_test.parquet'
TARGET_COL    = "부실라벨_ICR3년"
YEAR_COL      = "회계년도"
COMPANY_COL   = "회사명"
RANDOM_STATE  = 42
THRESHOLD     = 0.44
SHAP_SAVE_DIR = r"16번. SHAP\지역"
os.makedirs(SHAP_SAVE_DIR, exist_ok=True)

# ── 데이터 로드 ───────────────────────────────────────────
train_full   = pd.read_parquet(TRAIN_PATH)
test         = pd.read_parquet(TEST_PATH)
y_train_full = train_full[TARGET_COL]
y_test       = test[TARGET_COL]
pos_weight   = (y_train_full == 0).sum() / (y_train_full == 1).sum()
all_data     = pd.concat([train_full, test], ignore_index=True)

# ── 피처 파일 ─────────────────────────────────────────────
feature_files = {
    "top55_dedup45": r"13번.피처셀렉션\M19_도매_소매업\lasso_features_top55--45.csv",
}

# ── feature_map 재구성 ────────────────────────────────────
feature_map = {}
for feature_name, feature_path in feature_files.items():
    df_feat        = pd.read_csv(feature_path)
    col_key        = "feature" if "feature" in df_feat.columns else df_feat.columns[0]
    raw_features   = df_feat[col_key].tolist()
    valid_features = [f for f in raw_features if f in train_full.columns]
    feature_map[feature_name] = valid_features
    print(f"[{feature_name}]  피처: {len(valid_features)}개")

print(f"pos_weight : {pos_weight:.4f}")
print(f"전체 데이터: {len(all_data)}행  |  기업 수: {all_data[COMPANY_COL].nunique()}개")
print("=" * 70)


# ============================================================
# 조회할 기업 × 회계년도 목록 ← ★ 여기만 수정하면 됨
# ============================================================

QUERY_LIST = [
    {"회사명": "현대코퍼레이션(주)", "회계년도": 2021},
    {"회사명": "농협경제지주주식회사", "회계년도": 2021},
    # {"회사명": "기업명", "회계년도": 연도},
]


# ============================================================
# 헬퍼 함수
# ============================================================

def make_model():
    return XGBClassifier(
        n_estimators=300, learning_rate=0.05, max_depth=4,
        subsample=0.8, colsample_bytree=0.8,
        eval_metric="aucpr",
        random_state=RANDOM_STATE, verbosity=0,
        scale_pos_weight=pos_weight
    )


def get_sample(company, year):
    mask   = (all_data[COMPANY_COL] == company) & (all_data[YEAR_COL] == year)
    result = all_data[mask]
    if len(result) == 0:
        print(f"  ⚠️  [{company} / {year}] 데이터 없음")
        return None
    if len(result) > 1:
        print(f"  ⚠️  [{company} / {year}] 중복 {len(result)}행 → 첫 번째 행 사용")
    return result.iloc[[0]]


def fix_minus(fig):
    """유니코드 마이너스(−) → 일반 하이픈(-) 치환"""
    for ax in fig.get_axes():
        for t in ax.get_xticklabels():
            t.set_text(t.get_text().replace("\u2212", "-"))
        ax.set_xticklabels([t.get_text() for t in ax.get_xticklabels()])
        for t in ax.get_yticklabels():
            t.set_text(t.get_text().replace("\u2212", "-"))
        ax.set_yticklabels([t.get_text() for t in ax.get_yticklabels()])
        for txt in ax.texts:
            txt.set_text(txt.get_text().replace("\u2212", "-"))


# ============================================================
# 피처셋별 루프
# ============================================================

for feature_name, use_features in feature_map.items():

    print(f"\n{'='*70}")
    print(f"[로컬 SHAP] Feature Set: {feature_name}  |  피처 수: {len(use_features)}개")
    print(f"{'='*70}")

    save_sub = os.path.join(SHAP_SAVE_DIR, feature_name)
    os.makedirs(save_sub, exist_ok=True)

    # ── 전체 train으로 모델 학습 ──────────────────────────
    imputer     = SimpleImputer(strategy="median")
    X_train_imp = pd.DataFrame(
        imputer.fit_transform(train_full[use_features]),
        columns=use_features
    )
    model = make_model()
    model.fit(X_train_imp, y_train_full)

    # ── SHAP explainer 준비 ───────────────────────────────
    explainer = shap.TreeExplainer(model)

    # ── 전체 데이터 imputation ────────────────────────────
    all_feat_imp = pd.DataFrame(
        imputer.transform(all_data[use_features]),
        columns=use_features,
        index=all_data.index
    )

    # ── 쿼리별 분석 ───────────────────────────────────────
    for query in QUERY_LIST:

        company = query["회사명"]
        year    = query["회계년도"]

        print(f"\n  [{company} / {year}년]")

        sample_raw = get_sample(company, year)
        if sample_raw is None:
            continue

        sample_idx = sample_raw.index[0]
        X_sample   = all_feat_imp.loc[[sample_idx], use_features]

        y_true = sample_raw[TARGET_COL].values[0]
        y_prob = model.predict_proba(X_sample)[0, 1]
        y_pred = int(y_prob >= THRESHOLD)

        print(f"    실제 라벨 : {'부실(1)' if y_true == 1 else '정상(0)'}  |  "
              f"예측 확률 : {y_prob:.4f}  |  "
              f"예측 라벨 : {'부실(1)' if y_pred == 1 else '정상(0)'}")

        sv           = explainer(X_sample)
        shap_vals_1d = sv.values[0]
        base_value   = sv.base_values[0]

        safe_company = company.replace("/", "_").replace(" ", "_")
        save_company = os.path.join(save_sub, f"{safe_company}_{year}")
        os.makedirs(save_company, exist_ok=True)

        # ── [1] Waterfall Plot ────────────────────────────
        # shap 호출 직전 폰트 설정 강제 재적용
        mpl.rcParams["font.family"]        = "Malgun Gothic" \
                                             if platform.system() == "Windows" \
                                             else "AppleGothic"
        mpl.rcParams["axes.unicode_minus"] = False

        plt.figure(figsize=(10, max(6, len(use_features) * 0.28)))
        shap.plots.waterfall(sv[0], max_display=20, show=False)

        # 마이너스 기호 강제 치환
        fix_minus(plt.gcf())

        plt.title(
            f"SHAP Waterfall — {company} ({year}년)\n"
            f"실제: {'부실' if y_true==1 else '정상'}  |  "
            f"예측확률: {y_prob:.4f}  |  임계값: {THRESHOLD}",
            fontsize=11
        )
        plt.tight_layout()
        plt.savefig(
            os.path.join(save_company, f"waterfall_{safe_company}_{year}.png"),
            dpi=150, bbox_inches="tight"
        )
        plt.close()
        print(f"    → waterfall_{safe_company}_{year}.png 저장 완료")

        # ── [2] SHAP 기여도 CSV ───────────────────────────
        shap_row = pd.DataFrame({
            "Feature"    : use_features,
            "Value"      : X_sample.iloc[0].values,
            "SHAP_Value" : shap_vals_1d,
        })
        shap_row["Direction"] = shap_row["SHAP_Value"].apply(
            lambda x: "부실기여(+)" if x > 0 else "정상기여(-)"
        )
        shap_row["abs_SHAP"] = shap_row["SHAP_Value"].abs()
        shap_row = shap_row.sort_values(
            "abs_SHAP", ascending=False
        ).reset_index(drop=True)
        shap_row["Rank"]       = shap_row.index + 1
        shap_row.insert(0, "회계년도", year)
        shap_row.insert(0, "회사명",   company)
        shap_row["base_value"] = round(base_value, 6)
        shap_row["pred_prob"]  = round(y_prob, 6)
        shap_row["pred_label"] = y_pred
        shap_row["true_label"] = int(y_true)
        shap_row["threshold"]  = THRESHOLD

        shap_row = shap_row[[
            "회사명", "회계년도",
            "Rank", "Feature", "Value", "SHAP_Value",
            "Direction", "abs_SHAP",
            "base_value", "pred_prob", "pred_label",
            "true_label", "threshold"
        ]]
        shap_row.to_csv(
            os.path.join(save_company, f"shap_local_{safe_company}_{year}.csv"),
            index=False, encoding="utf-8-sig"
        )
        print(f"    → shap_local_{safe_company}_{year}.csv 저장 완료")

        # 상위 5개 출력
        print(f"\n    [상위 5개 기여 피처]")
        print(shap_row[[
            "Rank", "Feature", "Value", "SHAP_Value", "Direction"
        ]].head(5).to_string(index=False))


print("\n" + "=" * 70)
print("로컬 SHAP 분석 전체 완료")
print(f"  저장 위치: {SHAP_SAVE_DIR}")
print("=" * 70)

[top55_dedup45]  피처: 45개
pos_weight : 25.6455
전체 데이터: 39908행  |  기업 수: 6106개

[로컬 SHAP] Feature Set: top55_dedup45  |  피처 수: 45개

  [현대코퍼레이션(주) / 2021년]
    실제 라벨 : 정상(0)  |  예측 확률 : 0.0068  |  예측 라벨 : 정상(0)
    → waterfall_현대코퍼레이션(주)_2021.png 저장 완료
    → shap_local_현대코퍼레이션(주)_2021.csv 저장 완료

    [상위 5개 기여 피처]
 Rank               Feature    Value  SHAP_Value Direction
    1         총자본영업이익률_diff 0.946766   -3.065651   정상기여(-)
    2               금융비용부담률 0.010241   -0.976261   정상기여(-)
    3                 자본잠식률 0.956641   -0.246841   정상기여(-)
    4            순이익률_ratio 0.731541   -0.229111   정상기여(-)
    5 매출액순이익률_diff_industry 0.638665    0.213954   부실기여(+)

  [농협경제지주주식회사 / 2021년]
    실제 라벨 : 정상(0)  |  예측 확률 : 0.4265  |  예측 라벨 : 정상(0)
    → waterfall_농협경제지주주식회사_2021.png 저장 완료
    → shap_local_농협경제지주주식회사_2021.csv 저장 완료

    [상위 5개 기여 피처]
 Rank       Feature    Value  SHAP_Value Direction
    1       금융비용부담률 0.009081   -0.911516   정상기여(-)
    2        매출액증가율 0.246854    0.253075   부실기여(+